Description: Run Cosmos T1 GGUF chat in Colab with cached installs and Drive model reuse.

# Run Cosmos T1 GGUF

Execution order:
1. Mount Drive
2. Install `llama-cpp-python` (from Drive wheel if available; fallback compile)
3. Run chat script
4. (Optional, one-time) build and save a wheel to Drive for faster future sessions


In [5]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)


Mounted at /content/drive


In [ ]:
##build and save wheel

# %%bash

# set -euo pipefail

# if [ -d /content/drive/MyDrive/wheels ]; then
  # WHEEL_DIR='/content/drive/MyDrive/wheels'
# elif [ -d '/content/drive/My Drive/wheels' ]; then
  # WHEEL_DIR='/content/drive/My Drive/wheels'
# else
  # WHEEL_DIR='/content/drive/MyDrive/wheels'
# fi
# mkdir -p "$WHEEL_DIR"

# echo 'Building wheel (one-time, slow) and saving to Drive cache...'
# CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip wheel llama-cpp-python -w "$WHEEL_DIR"
# echo "Saved wheels under: $WHEEL_DIR"


In [6]:
%%bash
#build and save wheel if does not exist + pip install from the wheel
set -euo pipefail

if [ -d /content/drive/MyDrive/wheels ]; then
  WHEEL_DIR='/content/drive/MyDrive/wheels'
elif [ -d '/content/drive/My Drive/wheels' ]; then
  WHEEL_DIR='/content/drive/My Drive/wheels'
else
  WHEEL_DIR='/content/drive/MyDrive/wheels'
  mkdir -p "$WHEEL_DIR"
fi

if ls "$WHEEL_DIR"/llama_cpp_python-*.whl >/dev/null 2>&1; then
  echo 'Installing llama-cpp-python from Drive wheel cache...'
  pip install -U "$WHEEL_DIR"/llama_cpp_python-*.whl
else
  echo 'No cached wheel found. Building/installing from source (slow)...'
  CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --no-cache-dir -U llama-cpp-python
fi


Installing llama-cpp-python from Drive wheel cache...
Processing ./drive/MyDrive/wheels/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
llama-cpp-python is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [ ]:
# Run GGUF chat
from pathlib import Path
import runpy
import sys

if Path('/content/drive/MyDrive/training-embedding').exists():
    project_root = Path('/content/drive/MyDrive/training-embedding')
elif Path('/content/drive/My Drive/training-embedding').exists():
    project_root = Path('/content/drive/My Drive/training-embedding')
else:
    raise FileNotFoundError('Could not find training-embedding under Drive root.')

script_path = project_root / 'run_cosmos_t1_gguf.py'
if not script_path.exists():
    raise FileNotFoundError(f'Missing script: {script_path}')

# Drive-local GGUF cache path to avoid downloading each Colab restart.
save_load_path = project_root / 'models' / 'gguf_cache'
save_load_path.mkdir(parents=True, exist_ok=True)

print(f'Running: {script_path}')
print(f'Using --save_load_path: {save_load_path}')
print('Using --use_gpu')

old_argv = sys.argv[:]
try:
    sys.argv = [
        str(script_path),
        '--save_load_path', str(save_load_path),
        '--use_gpu',
    ]
    runpy.run_path(str(script_path), run_name='__main__')
finally:
    sys.argv = old_argv


Running: /content/drive/MyDrive/training-embedding/run_cosmos_t1_gguf.py
Using --save_load_path: /content/drive/MyDrive/training-embedding/models/gguf_cache
Using --use_gpu
Backend mode: GPU offload enabled (n_gpu_layers=-1)
Loading cached GGUF: /content/drive/MyDrive/training-embedding/models/gguf_cache/Turkish-Gemma-9b-T1.Q4_K_M.gguf


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized
llama_kv_cache_unified_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


History file not found, starting fresh: cosmos_t1_chat_history.json
Chat started. Commands: /reset, /exit
You: Merhaba nasılsın?
Assistant: Merhaba! 😊 Ben bir yapay zeka asistanıyım, bu yüzden duygulara sahip değilim ama seninle sohbet etmek ve yardımcı olmak için buradayım!  

Sen nasılsın? İyi misin? Bir konuda yardımına ihtiyacın var mı, yoksa sadece sohbet mi etmek istersin? 💬

You: Gökyüzü neden mavidir?
Assistant: Merhaba! 😊 Gökyüzünün mavi görünmesinin nedeni **atmosferdeki moleküllerin güneş ışığını saçmasıdır** (Rayleigh Saçılımı). İşte basit açıklaması:

### 1. **Güneş Işığı Beyazdır, Ama Renk İçerir**:  
   Güneş'ten gelen beyaz ışık aslında **tüm renklerin karışımıdır**. Bu ışık Dünya atmosferine girdiğinde, hava molekülleri (özellikle **azot ve oksijen**) tarafından **saçılır**.

### 2. **Mavi Işık Daha Çok Saçılır**:  
   - Kısa dalga boylu ışınlar (**mavi/mor**) uzun dalga boylulara (**kırmızı/sarı**) göre **daha güçlü saçılır**.  
   - Mavi ışık, mor ışıktan daha fazla 